<a href="https://colab.research.google.com/github/NehaBongarde2004/GenAI/blob/main/Exp8GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langgraph langchain openai


In [ ]:
import os
from openai import OpenAI

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    user_input: str
    decision: str
    response: str


In [ ]:
def input_node(state: AgentState):
    return state

def router_node(state: AgentState):
    user_input = state["user_input"]
    if "search" in user_input.lower():
        return {"decision": "tool"}
    else:
        return {"decision": "llm"}

def tool_node(state: AgentState):
    query = state["user_input"]
    result = f"Search results for: {query}"
    return {"response": result}

def llm_node(state: AgentState):
    user_input = state["user_input"]
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # Groq model
        messages=[{"role": "user", "content": user_input}]
    )
    answer = response.choices[0].message.content
    return {"response": answer}

def final_node(state: AgentState):
    return {"response": state.get("response", "No response generated")}


In [ ]:
graph = StateGraph(AgentState)

graph.add_node("input", input_node)
graph.add_node("router", router_node)
graph.add_node("tool", tool_node)
graph.add_node("llm", llm_node)
graph.add_node("final", final_node)

graph.set_entry_point("input")
graph.add_edge("input", "router")

# Conditional routing based on "decision" key
graph.add_conditional_edges(
    "router",
    lambda state: state["decision"],   # <- must return string key
    {"tool": "tool", "llm": "llm"}
)

graph.add_edge("tool", "final")
graph.add_edge("llm", "final")

app = graph.compile()


In [ ]:
models = client.models.list()
model_ids = [model.id for model in models.data]
print("Available Groq Models:")
for model_id in model_ids:
    print(f"- {model_id}")

Available Groq Models:
- groq/compound
- canopylabs/orpheus-v1-english
- whisper-large-v3
- openai/gpt-oss-20b
- openai/gpt-oss-120b
- whisper-large-v3-turbo
- meta-llama/llama-prompt-guard-2-86m
- llama-3.3-70b-versatile
- qwen/qwen3-32b
- meta-llama/llama-prompt-guard-2-22m
- openai/gpt-oss-safeguard-20b
- llama-3.1-8b-instant
- allam-2-7b
- meta-llama/llama-4-scout-17b-16e-instruct
- canopylabs/orpheus-arabic-saudi
- groq/compound-mini


In [ ]:
print(app.invoke({"user_input": "Explain LangGraph framework for agent workflows"}))
print(app.invoke({"user_input": "Explain LangGraph"}))


{'user_input': 'Explain LangGraph framework for agent workflows', 'decision': 'llm', 'response': "LangGraph is a Python framework specifically designed to develop and execute multi-agent workflows. It provides an easy-to-use interface for creating, managing, and executing agent workflows, making it suitable for various applications in domains like artificial intelligence, robotics, and cybersecurity.\n\nHere's an overview of LangGraph and its key features:\n\n**Key Components:**\n\n1. **Graph Definition**: LangGraph uses a graph-based representation to define workflows. This graph consists of nodes (agent tasks) and edges (data dependencies) that interact with each other to form a workflow.\n2. ** Agents (Nodes)**: Within a LangGraph, each node represents an autonomous agent or task that performs a specific function, such as data processing, communication, or control logic.\n3. ** Edges (Dependencies)**: Data dependencies between agents are represented by edges. An edge between two nod